# ISU-GeoBot — results, read live from the database

Every number in Chapter 4, printed from the rows it was computed from.

**Read-only.** Every statement in this notebook is a `SELECT`. Nothing here can
alter the evidence it displays, and nothing is recomputed — if a cell printed
something different from the paper, the paper would be wrong.

Run the cells in order. Each one answers one question a panel is likely to ask.

In [ ]:
import json, sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd() if (Path.cwd() / "database_connector.py").exists()
                       else Path.cwd() / "machine-learning"))
import database_connector as db

RUN   = "b0011d70-68b1-4f81-8d80-ef3f9e0d4acc"   # run-03-simulation
MODEL = "rf-20260909-072845"                     # the served classifier

pd.set_option("display.max_colwidth", 90)

def q(sql, params=()):
    return pd.DataFrame(db.fetch_all(sql, params))

print("connected \u2713")

## 1 · Where these numbers come from

> *"How do we know this is a real evaluation and not a demo?"*

The run is stored. Note the **judge model is not the generator** — the answers were
graded by a different model from the one that wrote them.

In [ ]:
run = q("""select run_label, started_at, prompt_template_version,
              groq_model_id, llm_temperature, embedding_model, top_k,
              similarity_floor, judge_model, status_as_context
       from geobot.eval_run where id = %s""", (RUN,))
display(run.T.rename(columns={0: "value"}))

counts = q("""select
     (select count(*) from geobot.eval_query)                        as registered_queries,
     (select count(*) from geobot.eval_result where run_id = %s)     as stored_results,
     (select count(*) from geobot.ragas_score g
        join geobot.eval_result r on r.id = g.eval_result_id
        where r.run_id = %s)                                          as scored_rows""",
   (RUN, RUN))
display(counts)

## 2 · SO2 (a) — the capability finding

> *"Which queries can the Enhanced architecture answer that retrieval alone cannot?"*

This is the headline: **0 of 6 → 5 of 6**. Not a quality difference — retrieval
reaches the document corpus, and a timetable is not a document.

In [ ]:
cap = q("""select r.mode,
          count(*) as asked,
          count(*) filter (where r.answer not ilike '%%sorry%%') as answered
       from geobot.eval_result r
       join geobot.eval_query q on q.id = r.eval_query_id
       where r.run_id = %s and q.category = 'faculty_availability'
       group by r.mode order by r.mode""", (RUN,))
display(cap)

### The six questions, side by side

The last row is the one to point at. **Both arms refuse it** — but the Enhanced
arm's classifier time is `0.0 ms`, so the estimate was *never computed*, not
computed and then hidden.

In [ ]:
six = q("""select q.query_text as question, r.mode, r.answer,
              r.t_rf_ms as classifier_ms
       from geobot.eval_result r
       join geobot.eval_query q on q.id = r.eval_query_id
       where r.run_id = %s and q.category = 'faculty_availability'
       order by q.query_text, r.mode""", (RUN,))

wide = six.pivot(index="question", columns="mode", values="answer")
wide["classifier_ms (enhanced)"] = (
    six[six["mode"] == "enhanced"].set_index("question")["classifier_ms"])
display(wide)

## 3 · SO2 (b) — the cost in response time

> *"What does the enhancement cost?"*

**Read this by category, never pooled.** The navigation row is the important one:
the classifier does *zero* work there, yet the measurement still shows a gap.
That gap is the instrument's noise floor — so any end-to-end difference of that
size carries no information about the architecture.

In [ ]:
t = q("""select q.category, r.mode,
          round(avg(r.t_total_ms)::numeric, 1) as total_ms,
          round(avg(r.t_rf_ms)::numeric, 1)    as classifier_ms,
          round(avg(r.t_guard_ms)::numeric, 1) as consent_gates_ms
       from geobot.eval_result r
       join geobot.eval_query q on q.id = r.eval_query_id
       where r.run_id = %s group by q.category, r.mode""", (RUN,))

tbl = t.pivot(index="category", columns="mode", values="total_ms")
tbl["difference"] = tbl["enhanced"] - tbl["standard"]
tbl["classifier_ms"] = t[t["mode"] == "enhanced"].set_index("category")["classifier_ms"]
tbl["gates_ms"] = t[t["mode"] == "enhanced"].set_index("category")["consent_gates_ms"]
display(tbl)

## 4 · SO2 (c) — the cost in RAGAS

> *"How did you compute these, and who graded them?"*

Four metrics, judged by `gpt-oss-20b` — **deliberately not the generator**. The
masked availability status is passed to RAGAS as a context item for the enhanced
arm, because it *is* retrieved context: retrieved from the classifier rather than
from pgvector. That decision is recorded on the run itself (`status_as_context`),
so it cannot be quietly changed after the fact.

**Paired means.** A query counts only where *both* arms produced a score, so the
two columns describe the same set of questions.

In [ ]:
def paired(metric):
    pair = f"""join (select r2.eval_query_id qid
        from geobot.ragas_score g2
        join geobot.eval_result r2 on r2.id = g2.eval_result_id
        where r2.run_id = %s and g2.{metric} is not null
        group by r2.eval_query_id having count(distinct r2.mode) = 2) p
        on p.qid = r.eval_query_id"""
    d = q(f"""select q.category, r.mode, round(avg(g.{metric})::numeric, 4) v, count(*) n
         from geobot.ragas_score g
         join geobot.eval_result r on r.id = g.eval_result_id
         join geobot.eval_query q on q.id = r.eval_query_id {pair}
         where r.run_id = %s and g.{metric} is not null
         group by q.category, r.mode""", (RUN, RUN))
    out = d.pivot(index="category", columns="mode", values="v").astype(float)
    out["n"] = d[d["mode"] == "enhanced"].set_index("category")["n"]
    return out

for m in ("faithfulness", "context_recall", "context_precision", "answer_relevancy"):
    print(m.replace("_", " ").title())
    display(paired(m))

**Say this before they spot it.** Context Precision is flat — it is a *retriever*
metric, and both arms share a retriever. That was predicted before it was measured.
Four metrics all rising would have been the suspicious result.

Standard scores exactly `0.0000` on availability because it produced no answer to grade.

## 5 · SO1 — the classifier

> *"What does the model look at, and how good is it?"*

**Say "simulation cohort" out loud whenever these numbers appear.**

In [ ]:
m = db.fetch_all("""select metrics, training_row_count, feature_list, algorithm,
                        class_order, split_strategy
                 from geobot.rf_model_version where version = %s""", (MODEL,))[0]
met = m["metrics"] if isinstance(m["metrics"], dict) else json.loads(m["metrics"])
fl  = m["feature_list"] if isinstance(m["feature_list"], list) else json.loads(m["feature_list"])

display(pd.DataFrame([{
    "model": MODEL,
    "algorithm": m["algorithm"],
    "accuracy": f"{met['accuracy'] * 100:.2f}%",
    "macro F1": f"{met['f1_macro']:.4f}",
    "cross-validation": f"{met['cv_f1_macro_mean']:.4f} +/- {met['cv_f1_macro_std']:.4f}",
    "train rows": f"{int(m['training_row_count']):,}",
    "test rows": f"{int(met['test_rows']):,}",
    "split": m["split_strategy"],
}]).T.rename(columns={0: "value"}))

display(pd.DataFrame(met["per_class"]).T[["precision", "recall", "f1", "support"]])

### The 11 features

Eight from the schedule, three from behaviour. The split is the argument: with
schedule features **and** schedule-derived labels, the forest would simply
reproduce the timetable lookup — it would *be* the rule baseline with a faculty
column. The three attendance features are what make it a different model.

In [ ]:
SCHEDULE = fl[:8]
ATTENDANCE = fl[8:]
display(pd.DataFrame({
    "schedule (8)":   SCHEDULE,
    "attendance (3)": ATTENDANCE + [""] * (len(SCHEDULE) - len(ATTENDANCE)),
}))

imp = db.fetch_all("select feature_importance from geobot.rf_model_version where version=%s",
                   (MODEL,))[0]["feature_importance"]
imp = imp if isinstance(imp, dict) else json.loads(imp)
s = pd.Series(imp).sort_values(ascending=False)
display(s.to_frame("mean Gini decrease").head(6))
print("\nTop feature is schedule-derived. Second is behavioural \u2014"
      " a timetable has no access to it.")